# CausalFlow DPO Fine-Tuning

Fine-tunes `meta-llama/Llama-3.2-3B-Instruct` on 231 CausalFlow-validated MBPP repair pairs using DPO (LoRA).  
After training, the model repairs failed agent steps **with no gold answer at inference**.

**To run this notebook, upload two files to your Google Drive root:**
- `mbpp_dpo_pairs.json`
- `causalflow_dpo.ipynb` (this file — open from Drive in Colab)

**Runtime:** A100 recommended (~2–3 hrs). T4 works with `LOAD_IN_4BIT = True` (~4–5 hrs).  
Go to: Runtime → Change runtime type → GPU

**Before running:** accept the LLaMA 3.2 license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct  
then add your HF token as a Colab Secret named `HF_TOKEN` (left panel → key icon).

## 1  —  Environment check

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → GPU (A100 or T4).')
print(result.stdout[:600])

vram_line = [l for l in result.stdout.splitlines() if 'MiB' in l]
print('VRAM line:', vram_line[0] if vram_line else 'unknown')

## 2  —  Install dependencies

In [ ]:
%%bash
pip install -q \
  "trl>=0.12" \
  "transformers>=4.45" \
  "peft>=0.13" \
  "accelerate>=0.34" \
  "bitsandbytes>=0.43" \
  "datasets>=2.20" \
  huggingface_hub

## 3  —  Mount Drive & authenticate

This mounts your Drive and reads `mbpp_dpo_pairs.json` from it.  
Outputs (checkpoints, eval results) are written back to Drive so they survive session restarts.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')  # set via Colab Secrets (left panel → key icon)
# or paste directly: HF_TOKEN = 'hf_...'

login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

## 4  —  Configuration

Change `DRIVE_ROOT` if you put the files in a subfolder, e.g. `MyDrive/causalflow/`.

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'         # change if files are in a subfolder
PAIRS_PATH   = f'{DRIVE_ROOT}/mbpp_dpo_pairs.json'
OUTPUT_DIR   = f'{DRIVE_ROOT}/causalflow_dpo_output'

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID     = 'meta-llama/Llama-3.2-3B-Instruct'
# MODEL_ID   = 'meta-llama/Llama-3.2-1B-Instruct'  # fallback for constrained VRAM

# ── Hardware ───────────────────────────────────────────────────────────────
LOAD_IN_4BIT = False   # set True for T4 (15 GB VRAM), keep False for A100

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 16
LORA_ALPHA   = 32

# ── DPO training ───────────────────────────────────────────────────────────
BETA         = 0.1
LR           = 5e-5
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
EPOCHS       = 3
MAX_LENGTH   = 1024
MAX_PROMPT   = 512

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sanity check
if not os.path.exists(PAIRS_PATH):
    raise FileNotFoundError(f'Cannot find {PAIRS_PATH} — make sure mbpp_dpo_pairs.json is in {DRIVE_ROOT}')
print(f'Found pairs file: {PAIRS_PATH}')
print(f'Outputs will be saved to: {OUTPUT_DIR}')

## 5  —  Load and inspect DPO pairs

In [ ]:
import json
from collections import defaultdict

pairs = json.load(open(PAIRS_PATH))
type_counts = defaultdict(int)
for p in pairs:
    type_counts[p['step_type']] += 1

scores = [p['minimality_score'] for p in pairs]
print(f'Total pairs     : {len(pairs)}')
print(f'Step types      : {dict(type_counts)}')
print(f'Minimality score: min={min(scores):.3f}  mean={sum(scores)/len(scores):.3f}  max={max(scores):.3f}')
print()
print('Example pair:')
ex = pairs[0]
print(f'  problem_id : {ex["problem_id"]}')
print(f'  step_type  : {ex["step_type"]}')
print(f'  rejected   : {ex["rejected"][:120]}...')
print(f'  chosen     : {ex["chosen"][:120]}...')

## 6  —  Stratified split and dataset formatting

In [ ]:
import random
from datasets import Dataset

def load_and_split(pairs, test_fraction=0.1, seed=42):
    by_type = defaultdict(list)
    for p in pairs:
        by_type[p['step_type']].append(p)
    rng = random.Random(seed)
    train, test = [], []
    for group in by_type.values():
        rng.shuffle(group)
        n_test = max(1, round(len(group) * test_fraction))
        test.extend(group[:n_test])
        train.extend(group[n_test:])
    return train, test

def format_pair(p):
    parts = [f"Problem: {p['problem_statement']}"]
    if p['prior_context'].strip():
        parts.append(f"Context: {p['prior_context'].strip()}")
    parts.append('Produce a corrected version of the following step:')
    return {'prompt': '\n'.join(parts), 'chosen': p['chosen'], 'rejected': p['rejected']}

train_pairs, test_pairs = load_and_split(pairs)
json.dump(test_pairs, open(f'{OUTPUT_DIR}/test_pairs.json', 'w'), indent=2)

train_dataset = Dataset.from_list([format_pair(p) for p in train_pairs])

tc = defaultdict(int)
for p in train_pairs: tc[p['step_type']] += 1
print(f'Train : {len(train_pairs)} {dict(tc)}')
tc = defaultdict(int)
for p in test_pairs: tc[p['step_type']] += 1
print(f'Test  : {len(test_pairs)} {dict(tc)}')
print(f'Test split saved → {OUTPUT_DIR}/test_pairs.json')

## 7  —  Load model and tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = None
torch_dtype = torch.bfloat16
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    torch_dtype = None

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch_dtype,
    device_map='auto',
    token=HF_TOKEN,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded — {total_params/1e9:.1f}B parameters')

## 8  —  Train

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
import re

def strip_fences(text):
    return re.sub(r'```\w*\n?', '', text).strip()

def format_sft(p):
    rejected_clean = strip_fences(p['rejected'])
    prompt_text = f"Problem: {p['problem_statement']}\n"
    if p['prior_context'].strip():
        prompt_text += f"Context: {p['prior_context'].strip()}\n"
    prompt_text += f"Failing step:\n{rejected_clean}\nOutput only the corrected code, no explanation:"
    messages = [
        {'role': 'user',      'content': prompt_text},
        {'role': 'assistant', 'content': p['chosen']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

sft_dataset = Dataset.from_list([format_sft(p) for p in train_pairs])

peft_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05, task_type='CAUSAL_LM',
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    bf16=not LOAD_IN_4BIT,
    fp16=False,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset,
    peft_config=peft_config,
)
trainer.train()

## 9  —  Save

Checkpoint is written directly to Drive (`OUTPUT_DIR`), so it persists even if the Colab session disconnects.

In [ ]:
FINAL_DIR = f'{OUTPUT_DIR}/final'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Saved to {FINAL_DIR}')

## 10  —  Evaluate (no gold at inference)

The model receives only the problem statement and prior trace context — no gold answer, no rejected step.  
Correctness is checked by running the generated code against MBPP test assertions via `exec()`.

In [ ]:
from peft import PeftModel

print('Loading fine-tuned model for evaluation ...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', token=HF_TOKEN
)
eval_model = PeftModel.from_pretrained(base_model, FINAL_DIR)
eval_model.eval()
print('Ready')

In [ ]:
from datasets import load_dataset

print('Loading MBPP test assertions ...')
mbpp = load_dataset('Muennighoff/mbpp', split='test+validation+train')
mbpp_tests = {row['task_id']: row['test_list'] for row in mbpp}
print(f'Loaded assertions for {len(mbpp_tests)} MBPP problems')

In [ ]:
def extract_code(text):
    if '```' in text:
        parts = text.split('```')
        for i, part in enumerate(parts):
            if i % 2 == 1:
                lines = part.strip().splitlines()
                if lines and not lines[0].strip().startswith(('def ', 'import ', 'class ', '#')):
                    lines = lines[1:]
                return '\n'.join(lines)
    return text.strip()

def exec_test(code, test_list):
    full = extract_code(code) + '\n' + '\n'.join(test_list)
    try:
        exec(compile(full, '<string>', 'exec'), {})
        return True
    except Exception:
        return False

def generate_repair(model, tokenizer, pair, max_new_tokens=512):
    parts = [f"Problem: {pair['problem_statement']}"]
    if pair['prior_context'].strip():
        parts.append(f"Context: {pair['prior_context'].strip()}")
    parts.append('Produce a corrected version of the following step:')
    prompt = '\n'.join(parts)
    # No rejected step in prompt — model must produce repair from context alone
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [ ]:
from difflib import SequenceMatcher

results = []
code_pass = code_total = 0

for pair in test_pairs:
    task_id = int(pair['problem_id'].replace('mbpp-', ''))
    tests = mbpp_tests.get(task_id, [])
    generated = generate_repair(eval_model, tokenizer, pair)

    entry = dict(pair)
    entry['generated'] = generated
    entry['passed'] = None

    if pair['step_type'] == 'llm_response' and tests:
        passed = exec_test(generated, tests)
        entry['passed'] = passed
        code_total += 1
        if passed:
            code_pass += 1
        print(f"  {pair['problem_id']} [code]      → {'PASS' if passed else 'FAIL'}")
    else:
        sim = SequenceMatcher(None, generated, pair['chosen']).ratio()
        entry['similarity_to_chosen'] = round(sim, 3)
        print(f"  {pair['problem_id']} [reasoning] → similarity={sim:.3f}")

    results.append(entry)

print('\nDone.')

In [ ]:
repair_rate = code_pass / code_total if code_total else 0.0
reasoning = [r for r in results if r['step_type'] == 'reasoning']
avg_sim = sum(r.get('similarity_to_chosen', 0) for r in reasoning) / len(reasoning) if reasoning else 0.0

print('══════════════════════════════════════════════════════')
print(f'Code repair rate  :  {code_pass}/{code_total} = {repair_rate:.1%}')
print(f'Reasoning sim avg :  {avg_sim:.3f} over {len(reasoning)} pairs')
print()
print('Comparison table (paper Section 6.6):')
rows = [
    ('CausalFlow (main — gold in prompt)',  'Yes', '44.9%'),
    ('No-gold ablation (zero-shot)',         'No',  '~0%'),
    ('DPO LLaMA 3B — this run',             'No',  f'{repair_rate:.1%}'),
]
print(f'{"Condition":<45} {"Gold":^6} {"Repair rate":>12}')
print('-' * 65)
for condition, gold, rate in rows:
    print(f'{condition:<45} {gold:^6} {rate:>12}')

out = {'summary': {'code_repair_rate': repair_rate, 'code_passed': code_pass,
                   'code_total': code_total, 'reasoning_avg_similarity': avg_sim},
       'details': results}
json.dump(out, open(f'{OUTPUT_DIR}/eval_results.json', 'w'), indent=2)
print(f'\nResults saved → {OUTPUT_DIR}/eval_results.json')